# Hotel and Homestay Review-Aware Recommender (MP-A13)

- **Course:** CS467 Advanced Artificial Intelligence (2026–2027) — Assignment 15
- **Student:** Nguyễn Ngọc Hoàng Thái (`29211151440`) — Section `CS467A (No. 13)`
- **Instructor:** Puri Vikram

This notebook walks through the dataset, NLP preprocessing & aspect-sentiment pipeline, recommendation engines (Popularity Baseline, NLP Content-Based, User-Based Collaborative Filtering, and Review-Aware Hybrid), offline evaluation, and all 3 required test cases.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from nlp_pipeline import StayNLPPipeline, analyze_review_aspects, detect_query_aspects
from evaluate import load_data, train_test_split_by_user, build_models, evaluate_models

items_df, ratings_df = load_data()
n_users = ratings_df['user_id'].nunique()
n_items = items_df['item_id'].nunique()
sparsity = (1 - len(ratings_df) / (n_users * n_items)) * 100
print(f'Stays: {n_items} | Guests: {n_users} | Ratings: {len(ratings_df)} | Sparsity: {sparsity:.2f}%')
display(items_df.head(5))
display(ratings_df.head(5))


## 1. NLP Preprocessing & Clause-Level Aspect-Sentiment Demo


In [ ]:
sample_reviews = [
    'Spotless room and super quiet at night near the beach. Strong wifi and friendly host!',
    'Stunning rooftop views and photogenic pool, but the bathroom was not clean, noisy street at night, and staff were unhelpful.'
]
for rev in sample_reviews:
    print('Review:', rev)
    print('Aspects:', analyze_review_aspects(rev))
    print()


## 2. Fit Full Recommender Models & Run the 3 Required Test Cases


In [ ]:
nlp, pop, cb, cf, hybrid = build_models(items_df, ratings_df)
print('All models fitted successfully!')


### Test Case 1: Stay Query (`quiet homestay near beach with strong wifi and friendly host`)


In [ ]:
q1 = 'quiet homestay near beach with strong wifi and friendly host'
print('Detected query aspect weights:', detect_query_aspects(q1))
print('\n--- NLP Content-Based (TF-IDF Cosine) Top 5 ---')
display(cb.recommend_by_query(q1, top_k=5))
print('\n--- Review-Aware Hybrid Top 5 ---')
display(hybrid.recommend_by_query(q1, top_k=5))


### Test Case 2: Review-Aware Ranking vs. Plain Star Average


In [ ]:
cols = ['plain_rank', 'review_aware_rank', 'rank_change', 'item_id', 'stay_name', 'avg_rating', 'overall_aspect_sentiment', 'review_aware_score']
print('Top 8 by Plain Star Average:')
display(hybrid.compare_plain_vs_review_aware(top_k=8, sort_by='plain')[cols])
print('Top 8 by Review-Aware Score (55% Star Avg + 45% Aspect Sentiment):')
display(hybrid.compare_plain_vs_review_aware(top_k=8, sort_by='review_aware')[cols])


### Test Case 3: Cold-Start Test (Brand-New Guest with Natural-Language Request)


In [ ]:
cold_query = 'peaceful heritage homestay near ancient town with spotless room, kind local host, and reliable wifi'
display(hybrid.recommend_by_query(cold_query, top_k=5, user_id='NEW_GUEST_01'))


## 3. Personalized Comparison for an Existing Guest (`U001`)


In [ ]:
print('Popularity Baseline:')
display(pop.recommend('U001', top_k=5))
print('User-Based Collaborative Filtering:')
display(cf.recommend('U001', top_k=5))
print('Review-Aware Hybrid:')
display(hybrid.recommend_for_user('U001', top_k=5))


## 4. Offline Train/Test Evaluation & Visualizations


In [ ]:
train_df, test_df = train_test_split_by_user(ratings_df, test_ratio=0.25, min_test_per_user=3, seed=42)
eval_df = evaluate_models(items_df, train_df, test_df, k=5, relevance_threshold=4.0)
display(eval_df)

for chart_name in ['data_distribution.png', 'model_comparison.png', 'aspect_sentiment_comparison.png']:
    chart_path = PROJECT_ROOT / 'results' / chart_name
    if chart_path.exists():
        display(Image(filename=str(chart_path)))
